# Kaufland CS Guardrail & Voice Agent - Demonstration Notebook
This notebook demonstrates the core components of our architecture: configuration validation, robust intent parsing, hybrid RAG retrieval, and telemetry logging.

In [2]:
import os
import asyncio
from dotenv import load_dotenv
import sys

load_dotenv()

# Verify environment variables
print(f"Groq API Key Set: {bool(os.getenv('GROQ_API_KEY'))}")
print(f"Deepgram API Key Set: {bool(os.getenv('DEEPGRAM_API_KEY'))}")

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

Groq API Key Set: True
Deepgram API Key Set: True


In [3]:
from src.chatbot import Config, GraphProcessor
config = Config()
processor = GraphProcessor(config)
print("Config and Graph Processor initialized successfully.")

[Guardrail Node] Initializing guardrails...
[Guardrails] Loading NLP models...
📚 [RAG Agent] Booting up database, guardrails, and hybrid retriever...
[RAG Engine] Initializing HuggingFace Embeddings...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[RAG Engine] Connecting to existing ChromaDB...
[RAG Engine] Ready!
[Guardrails] Loading NLP models...
📚 [RAG Agent] Hybrid retriever ready with 451 documents indexed.
Config and Graph Processor initialized successfully.


## Testing the Intent Agent 
Test how the system classifies user input and routes it between RAG, small talk, and escalation without crashing on Groq token limits.

In [7]:
# Cell 4: Execute Test Turns (Jupyter supports top-level await natively)
demo_queries = [
    "Hallo, guten Tag!",
    "Was ist Kaufland Pay?",
    "Wie lautet die interne Steuerungsnummer für Filiale 9999?"
]

for query in demo_queries:
    print(f"\n--- User Query: '{query}' ---")
    response = await processor.generate_response(query)
    print(f"Action Route : {response['action'].upper()}")
    print(f"Confidence   : {response['confidence']}")
    print(f"Latency      : {response['elapsed_ms']} ms")
    print(f"🤖 Bot Response : {response['text']}")
    await asyncio.sleep(0.5)


--- User Query: 'Hallo, guten Tag!' ---
[Guardrail Node] Validating input...
✅ [Guardrail Node] Input is safe. Proceeding.
[Intent Agent] Analyzing customer message...
[Intent Agent] Decision made: ANSWER
👋 [Direct Response] Small talk detected, replying directly.
Action Route : ANSWERED
Confidence   : 0.0
Latency      : 480 ms
🤖 Bot Response : Hallo! Wie kann ich Ihnen heute mit Ihrer Kaufland-Frage helfen?

--- User Query: 'Was ist Kaufland Pay?' ---
[Guardrail Node] Validating input...
✅ [Guardrail Node] Input is safe. Proceeding.
[Intent Agent] Analyzing customer message...
[Intent Agent] Decision made: RAG
📚 [RAG Agent] Searching for answers...
📚 [RAG Agent] Query corrected: 'Was ist Kaufland Pay?' -> 'was ist kaufland pay'
📚 [RAG Agent] Answer generated.
[Confidence Agent] Inspecting the generated answer...
[Confidence Agent] Grade: 1.0 - The answer accurately reflects the retrieved facts and directly addresses the question without adding unsupported information.
Action Route : 